# Python Functions — Deep Dive

A hands-on, runnable reference dedicated entirely to Python functions — from the basics of `def` all the way to closures, decorators, and generators. `Python/basics.ipynb` introduces functions briefly; this notebook goes much deeper. Run each cell as you go.

In [ ]:
def greet():
    print("Hello!")

greet()

## 1. Defining and Calling Functions

- A function is defined with `def name(parameters):` and a block of indented code.
- Calling it is just `name(arguments)`.
- A function with no explicit `return` implicitly returns `None`.
- Functions must be **defined before** they're called (top-to-bottom execution) — except when both are inside other functions/classes evaluated later.

In [ ]:
def square(n):
    return n * n

print(square(5))

# No explicit return -> implicitly returns None
def log_message(msg):
    print(f"[LOG] {msg}")

result = log_message("something happened")
print(result)   # None

# A function is just another object - it has a type, an id, a __name__...
print(type(square), square.__name__)

## 2. Parameters vs Arguments

- **Parameters** are the names listed in the function definition; **arguments** are the actual values passed when calling.
- Arguments can be passed **positionally** (matched by order) or **by keyword** (matched by name) — you can mix both, but positional arguments must come first.
- Passing too few/many arguments, an unknown keyword, or the same argument twice raises a `TypeError`.

In [ ]:
def describe_pet(animal, name, age):
    print(f"{name} is a {age}-year-old {animal}")

# All positional
describe_pet("dog", "Rex", 3)

# All keyword - order doesn't matter
describe_pet(name="Whiskers", age=2, animal="cat")

# Mixed - positional first, then keyword
describe_pet("dog", name="Buddy", age=5)

# Common errors
try:
    describe_pet("dog", "Rex")              # missing required argument
except TypeError as e:
    print("TypeError:", e)

try:
    describe_pet("dog", "Rex", 3, breed="Lab")   # unexpected keyword argument
except TypeError as e:
    print("TypeError:", e)

## 3. Default Parameter Values

- A parameter can have a default value (`def f(x=10):`), making it optional.
- Parameters with defaults must come **after** parameters without defaults.
- **Danger:** a mutable default (`[]`, `{}`) is created **once**, at function-definition time, and reused across every call that doesn't override it — a very common bug.

In [ ]:
def power(base, exponent=2):
    return base ** exponent

print(power(5))         # uses default exponent=2
print(power(5, 3))       # overrides the default

# Defaults after non-defaults is a SyntaxError:
# def bad(x=1, y): ...

# The mutable default argument trap
def add_item(item, basket=[]):     # DANGER: this list is created ONCE
    basket.append(item)
    return basket

print(add_item("apple"))    # ['apple']
print(add_item("banana"))   # ['apple', 'banana'] <- surprising! same list reused

# Correct pattern: default to None, create the mutable object inside the function
def add_item_fixed(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item_fixed("apple"))
print(add_item_fixed("banana"))

## 4. Variable-Length Arguments: `*args` and `**kwargs`

- `*args` collects any extra **positional** arguments into a `tuple`.
- `**kwargs` collects any extra **keyword** arguments into a `dict`.
- Naming them `args`/`kwargs` is convention only — the `*`/`**` is what matters.
- Order in the signature: normal params → `*args` → keyword-only params → `**kwargs`.

In [ ]:
def total(*numbers):
    print(type(numbers), numbers)
    return sum(numbers)

print(total(1, 2, 3))
print(total(1, 2, 3, 4, 5))
print(total())              # zero args is fine too - numbers is just ()

def build_profile(name, **details):
    print(type(details), details)
    return {"name": name, **details}

profile = build_profile("Alice", age=30, city="NYC")
print(profile)

# Combining everything in one signature
def full_example(required, *args, default=10, **kwargs):
    print("required:", required)
    print("args:", args)
    print("default:", default)
    print("kwargs:", kwargs)

full_example(1, 2, 3, default=99, extra="hi")

## 5. Positional-Only and Keyword-Only Parameters

- A `/` in the signature marks everything **before** it as **positional-only** — callers cannot use those names as keywords.
- A `*` (or `*args`) marks everything **after** it as **keyword-only** — callers must use those names explicitly.
- This is used heavily in the standard library to lock down a stable calling convention.

In [ ]:
def example(pos_only, /, normal, *, kw_only):
    return pos_only, normal, kw_only

print(example(1, 2, kw_only=3))          # OK
print(example(1, normal=2, kw_only=3))   # OK - 'normal' can be positional OR keyword

try:
    example(pos_only=1, normal=2, kw_only=3)   # pos_only can't be passed by keyword
except TypeError as e:
    print("TypeError:", e)

try:
    example(1, 2, 3)   # kw_only MUST be passed by keyword
except TypeError as e:
    print("TypeError:", e)

## 6. Return Values

- `return` immediately exits the function with a value; a bare `return` (or falling off the end) returns `None`.
- Returning several values separated by commas actually returns a single **tuple**, which the caller can unpack.
- A function can have multiple `return` statements — the first one reached wins.

In [ ]:
def min_max(numbers):
    return min(numbers), max(numbers)   # this is really `return (min(numbers), max(numbers))`

result = min_max([4, 2, 9, 1])
print(result, type(result))

lo, hi = min_max([4, 2, 9, 1])    # unpacked directly
print(lo, hi)

def classify(n):
    if n < 0:
        return "negative"
    if n == 0:
        return "zero"
    return "positive"          # only reached if both ifs above were False

for n in [-5, 0, 5]:
    print(n, "->", classify(n))

## 7. Docstrings and Introspection

- A **docstring** is a string literal as the first statement in a function body — it becomes `func.__doc__` and is what `help()` displays.
- Functions are objects, so you can inspect their name, defaults, and annotations at runtime via `__name__`, `__defaults__`, `__annotations__`, and the `inspect` module.

In [ ]:
def divide(a, b):
    """Return a divided by b.

    Raises ZeroDivisionError if b is 0.
    """
    return a / b

print(divide.__doc__)
help(divide)

import inspect
print(inspect.signature(divide))
print(inspect.getsource(divide).splitlines()[0])

## 8. Type Hints (Annotations)

- Type hints document the expected parameter/return types (`def f(x: int) -> str:`) but Python does **not** enforce them at runtime — they're for readers, IDEs, and tools like `mypy`.
- Hints are stored in `func.__annotations__`.

In [ ]:
def greet(name: str, times: int = 1) -> str:
    return f"Hello, {name}! " * times

print(greet("Alice", 2))
print(greet.__annotations__)

# Python does NOT enforce these at runtime - "wrong" types are silently accepted.
# name=123 is an int, not the str the hint promises, yet this runs without complaint:
# the f-string happily coerces it to text, so nothing catches the type mismatch.
print(greet(123, 2))

## 9. Variable Scope in Functions

- Each function call gets its own **local** namespace; variables created inside are invisible outside.
- Python resolves names via **LEGB**: Local → Enclosing → Global → Built-in.
- `global` lets a function **assign to** a global variable (not just read it); `nonlocal` does the same for an enclosing (non-global) function's variable.

In [ ]:
x = "global value"

def read_only():
    print(x)     # reading a global works without any keyword

read_only()

def local_shadow():
    x = "local value"    # this creates a NEW local variable, doesn't touch the global
    print(x)

local_shadow()
print(x)   # unchanged

counter = 0
def increment():
    global counter          # without this, the next line would raise UnboundLocalError
    counter += 1

increment(); increment(); increment()
print(counter)

def outer():
    total = 0
    def inner():
        nonlocal total       # refers to outer()'s 'total', not a new local one
        total += 1
    inner(); inner()
    return total

print(outer())

## 10. Recursion

A recursive function calls itself, working toward a **base case** that stops the recursion. Python has a default recursion depth limit (`sys.getrecursionlimit()`, usually 1000) — deep recursion raises `RecursionError`; an iterative or memoized approach is often safer for large inputs.

In [ ]:
def factorial(n):
    if n <= 1:          # base case
        return 1
    return n * factorial(n - 1)   # recursive case

print(factorial(6))

def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print([fibonacci(i) for i in range(10)])

import sys
print("recursion limit:", sys.getrecursionlimit())

def infinite_recursion(n):
    return infinite_recursion(n + 1)

try:
    infinite_recursion(0)
except RecursionError as e:
    print("RecursionError:", e)

## 11. Lambda Functions

A `lambda` is a small, single-expression, **anonymous** function — `lambda args: expression` is equivalent to a `def` whose body is just `return expression`. Useful for short, throwaway functions passed to things like `sorted()`, `map()`, `filter()`.

In [ ]:
square = lambda x: x ** 2
print(square(5))

add = lambda a, b: a + b
print(add(3, 4))

# Equivalent def, for comparison
def square_def(x):
    return x ** 2
print(square_def(5) == square(5))

# Most common use: as a throwaway "key" function
words = ["banana", "kiwi", "apple", "fig"]
print(sorted(words, key=lambda w: len(w)))
print(sorted(words, key=len))   # even simpler when a builtin already does the job

## 12. Higher-Order Functions

Because functions are first-class objects, they can be **passed as arguments** and **returned as values** from other functions — a function that does either is a "higher-order function". `map()`, `filter()`, and `functools.reduce()` are the classic built-in examples.

In [ ]:
from functools import reduce

nums = [1, 2, 3, 4, 5]

print(list(map(lambda x: x * 2, nums)))              # apply a function to every item
print(list(filter(lambda x: x % 2 == 0, nums)))       # keep items where the function is True
print(reduce(lambda acc, x: acc + x, nums))            # combine items into a single value
print(reduce(lambda acc, x: acc * x, nums, 1))          # with an explicit start value

# A function that TAKES a function as an argument
def apply_twice(func, value):
    return func(func(value))

print(apply_twice(lambda x: x + 3, 10))   # 10 -> 13 -> 16

# A function that RETURNS a function
def make_multiplier(factor):
    def multiplier(x):
        return x * factor
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5), triple(5))

## 13. Closures

A **closure** is an inner function that "remembers" variables from its enclosing function's scope, even after the outer function has finished running. `make_multiplier` above is already a closure — `multiplier` keeps a reference to `factor` long after `make_multiplier()` returned.

In [ ]:
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

c1 = make_counter()
c2 = make_counter()   # a completely independent closure, its own 'count'

print(c1(), c1(), c1())   # 1 2 3
print(c2())                 # 1 - unaffected by c1

# Inspecting what a closure captured
print(c1.__closure__[0].cell_contents)

# Classic gotcha: closures capture variables BY REFERENCE, not by value
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])          # [2, 2, 2] - all three share the same 'i'

funcs_fixed = [lambda i=i: i for i in range(3)]   # default arg captures the value NOW
print([f() for f in funcs_fixed])    # [0, 1, 2]

## 14. Decorators

A decorator is a function that **wraps another function**, adding behavior before/after it runs, without modifying its source code. `@decorator` above a `def` is shorthand for `func = decorator(func)`. `functools.wraps` preserves the original function's name/docstring on the wrapped version.

In [ ]:
import functools
import time

def log_calls(func):
    @functools.wraps(func)         # preserves func.__name__ and __doc__
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

@log_calls
def add(a, b):
    """Add two numbers."""
    return a + b

print(add(3, 4))
print(add.__name__, "-", add.__doc__)   # preserved thanks to functools.wraps

# A decorator that takes its OWN arguments needs an extra layer of nesting
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            results = [func(*args, **kwargs) for _ in range(times)]
            return results
        return wrapper
    return decorator

@repeat(times=3)
def shout(word):
    return word.upper() + "!"

print(shout("hi"))

# A simple timing decorator
def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timed
def slow_square(n):
    return n * n

print(slow_square(12))

## 15. Generator Functions

A function containing `yield` becomes a **generator function** — calling it doesn't run the body immediately, it returns a generator object that produces values **lazily**, one at a time, pausing at each `yield` and resuming on the next `next()` call. This uses far less memory than building a full list upfront.

In [ ]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

gen = count_up_to(5)
print(type(gen))
print(next(gen), next(gen))
print(list(gen))    # consumes the REST of the generator (3, 4, 5) - already-yielded values are gone

for value in count_up_to(3):    # a fresh generator each call
    print(value)

# Generators are memory-efficient for large/infinite sequences
def infinite_counter():
    n = 0
    while True:
        yield n
        n += 1

counter = infinite_counter()
first_five = [next(counter) for _ in range(5)]
print(first_five)   # only 5 values were ever computed, despite the "infinite" loop

# A generator can also receive values back via .send(), and return a final value
def running_total():
    total = 0
    while True:
        amount = yield total
        if amount is not None:
            total += amount

acc = running_total()
next(acc)                 # prime the generator (advance to the first yield)
print(acc.send(10))
print(acc.send(5))

## 16. The `functools` Toolkit

- `functools.partial()` pre-fills some arguments of a function, producing a new, simpler callable.
- `functools.lru_cache()` memoizes a function's results, dramatically speeding up repeated/recursive calls with the same arguments.
- `functools.singledispatch` lets a function behave differently based on the type of its first argument ("overloading" by type).

In [ ]:
from functools import partial, lru_cache, singledispatch

# partial: lock in some arguments ahead of time
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)
cube = partial(power, exponent=3)
print(square(5), cube(5))

# lru_cache: memoize expensive/recursive calls
call_count = 0

@lru_cache(maxsize=None)
def fib(n):
    global call_count
    call_count += 1
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30))
print("fib(30) took only", call_count, "actual calls thanks to caching")

# singledispatch: pick an implementation based on the argument's type
@singledispatch
def describe(value):
    return f"Some value: {value}"

@describe.register
def _(value: int):
    return f"An integer: {value}"

@describe.register
def _(value: str):
    return f"A string of length {len(value)}: {value!r}"

print(describe(42))
print(describe("hello"))
print(describe(3.14))

## 17. Unpacking Arguments at the Call Site

`*` and `**` aren't just for *defining* flexible signatures — you can use them when *calling* a function too, to spread a list/tuple into positional arguments or a dict into keyword arguments.

In [ ]:
def describe_point(x, y, z):
    return f"({x}, {y}, {z})"

coords = [1, 2, 3]
print(describe_point(*coords))     # same as describe_point(1, 2, 3)

coords_dict = {"x": 1, "y": 2, "z": 3}
print(describe_point(**coords_dict))   # same as describe_point(x=1, y=2, z=3)

# Very useful for forwarding arguments through a wrapper
def forwarder(*args, **kwargs):
    return describe_point(*args, **kwargs)

print(forwarder(4, 5, z=6))

## 18. Best Practices

- **Single responsibility**: a function should do one thing well and be named after what it does.
- **Prefer pure functions** where practical — same input always gives the same output, no hidden side effects (no mutating arguments, no reading/writing global state). Pure functions are trivial to test and reason about.
- **Avoid mutable default arguments** (see section 3).
- **Keep signatures small**; if a function needs many related parameters, consider grouping them into a single object/dataclass.
- **Write a docstring** for anything non-trivial — it doubles as documentation and `help()` output.

In [ ]:
# Impure: depends on and mutates external state - hard to test in isolation
total = 0
def add_to_total(n):
    global total
    total += n
    return total

# Pure: same input always gives the same output, no side effects
def add(a, b):
    return a + b

print(add(2, 3) == add(2, 3) == 5)   # always true, guaranteed
print(add_to_total(5), add_to_total(5))   # NOT the same result twice - depends on prior calls

# Grouping many related parameters into one object beats a long positional list
from dataclasses import dataclass

@dataclass
class Rectangle:
    width: float
    height: float

def area(rect: Rectangle) -> float:
    """Return the area of a rectangle."""
    return rect.width * rect.height

print(area(Rectangle(width=4, height=5)))